In [9]:
# =====================================================================
# IMPORT MODULES & ENVIRONMENT SETUP
# =====================================================================
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

# Import warehouse constants from the environment module
from warehouse_env import warehouse_positions, io_point

# Load the clustered dataset (Ensure this path matches your directory)
df_final = pd.read_csv('../../features/sku_clusters_final.csv')

# Helper function to calculate Manhattan distance
def get_manhattan_dist(p1, p2):
    """Calculates the distance between two (x, y) coordinates."""
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

print(f"Warehouse environment initialized with {len(warehouse_positions)} slots.")
print(f"Total SKUs loaded: {len(df_final)}")

Warehouse environment initialized with 4250 slots.
Total SKUs loaded: 4130


In [13]:
# =====================================================================
# REAL-WORLD ORDER SIMULATION (BASED ON INVOICE DATA)
# =====================================================================
import pandas as pd
import numpy as np

# 1. Load data_clean to extract real historic orders
# Make sure the path correctly points to your data_clean.csv file
df_orders = pd.read_csv('../../data/processed/data_clean.csv') 

# 2. Get the set of valid SKUs that exist in our clustering results
# This ensures consistency and prevents missing key errors during lookup
valid_skus = set(df_final['stock_code'].unique())

# 3. Filter the historic dataset to keep only clustered SKUs
df_valid_orders = df_orders[df_orders['stock_code'].isin(valid_skus)]

# 4. Group by invoice_no to reconstruct real-world customer baskets
# Filter out single-item orders to focus on multi-item picking efficiency
real_orders = df_valid_orders.groupby('invoice_no')['stock_code'].apply(list)
real_orders = [order for order in real_orders if len(order) >= 2]

print(f"Successfully loaded {len(real_orders):,} real multi-item orders from transaction history.")

# 5. Define the picking simulation function
def simulate_order(order, layout):
    dist = 0
    curr = io_point
    for sku in order:
        if sku in layout:
            target = layout[sku]
            dist += get_manhattan_dist(curr, target)
            curr = target
    dist += get_manhattan_dist(curr, io_point) # Return to the I/O point
    return dist

# ---------------------------------------------------------------------
# BUILD LAYOUTS: K-MEANS, ABC, RANDOM
# ---------------------------------------------------------------------
# Sort warehouse positions by Manhattan distance from the I/O point
positions_sorted = sorted(warehouse_positions, key=lambda p: get_manhattan_dist(p, io_point))

# 1) K-Means layout: RE-EVALUATE AND RANK CLUSTERS
# Calculate average pick frequency per Cluster to identify the high-demand ("VIP") group
cluster_mean_freq = df_final.groupby('Cluster')['pick_frequency'].mean().sort_values(ascending=False).index

# Map clusters to ranks: The cluster with the highest frequency gets rank 0 (closest to I/O point)
cluster_rank_mapping = {cluster: rank for rank, cluster in enumerate(cluster_mean_freq)}
df_final['cluster_rank'] = df_final['Cluster'].map(cluster_rank_mapping)

# Hybrid Sort: Sort by cluster rank first (ascending), then by pick frequency (descending) within the cluster
df_k = df_final.sort_values(['cluster_rank', 'pick_frequency'], ascending=[True, False])
stock_list_k = df_k['stock_code'].tolist()
layout_kmeans = {sku: positions_sorted[i] for i, sku in enumerate(stock_list_k)}

# 2) ABC layout: place 'A' items closest, then 'B', then 'C'
abc_order = {'A': 0, 'B': 1, 'C': 2}
df_abc_sorted = df_final.assign(abc_rank=df_final['abc_class'].map(abc_order))
df_abc_sorted = df_abc_sorted.sort_values(['abc_rank', 'pick_frequency'], ascending=[True, False])
stock_list_abc = df_abc_sorted['stock_code'].tolist()
layout_abc = {sku: positions_sorted[i] for i, sku in enumerate(stock_list_abc)}

# 3) Random layout: shuffle SKU list and assign positions
stock_list = df_final['stock_code'].tolist()
random.seed(42)
random.shuffle(stock_list)
layout_random = {sku: positions_sorted[i] for i, sku in enumerate(stock_list)}

print('Layouts created:', len(layout_kmeans), len(layout_abc), len(layout_random))

# 6. Execute picking simulation across all 3 layouts using historic data
res_kmeans = [simulate_order(o, layout_kmeans) for o in real_orders]
res_abc = [simulate_order(o, layout_abc) for o in real_orders]
res_random = [simulate_order(o, layout_random) for o in real_orders]

# Report how many orders were processed
total_orders = len(real_orders)
processed_orders = len(res_kmeans)
print(f"Processed {processed_orders:,}/{total_orders:,} orders")

# 7. Print final performance metrics for comparison
print(f"Average Picking Distance (K-Means): {np.mean(res_kmeans):.2f} units")
print(f"Average Picking Distance (ABC):     {np.mean(res_abc):.2f} units")
print(f"Average Picking Distance (Random):  {np.mean(res_random):.2f} units")

Successfully loaded 18,778 real multi-item orders from transaction history.
Layouts created: 4090 4090 4090
Processed 18,778/18,778 orders
Average Picking Distance (K-Means): 799.83 units
Average Picking Distance (ABC):     806.93 units
Average Picking Distance (Random):  1276.93 units


In [15]:
# =====================================================================
# EVALUATE EFFICIENCY (TRAVEL TIME REDUCTION)
# =====================================================================
import pandas as pd

# 1. Define Warehouse Assumptions
WALKING_SPEED_MPS = 1.2  # Speed in meters per second (human with picking cart)
SLOT_SIZE_METERS = 1.0   # 1 distance unit = 1 meter

def calculate_time_hours(distance_list):
    """Converts a list of order distances into total travel time in hours."""
    total_distance_meters = sum(distance_list) * SLOT_SIZE_METERS
    total_seconds = total_distance_meters / WALKING_SPEED_MPS
    total_hours = total_seconds / 3600  # Convert seconds to hours
    return total_hours

# 2. Calculate Total Travel Time for all 18,778 orders
time_kmeans = calculate_time_hours(res_kmeans)
time_abc = calculate_time_hours(res_abc)
time_random = calculate_time_hours(res_random)

# 3. Create a Summary DataFrame for reporting
summary_data = {
    "Slotting Strategy": ["Random Layout", "ABC Layout", "K-Means Layout"],
    "Total Distance (Meters)": [sum(res_random), sum(res_abc), sum(res_kmeans)],
    "Total Travel Time (Hours)": [time_random, time_abc, time_kmeans]
}

df_summary = pd.DataFrame(summary_data)

# 4. Print the Business Report
print("=======================================================")
print("       EFFICIENCY EVALUATION (TRAVEL TIME REPORT)      ")
print("=======================================================")
print(df_summary.to_string(index=False, float_format="%.2f"))
print("=======================================================\n")

# Calculate metrics compared to Random Baseline
saved_hours_vs_random = time_random - time_kmeans
improvement_pct_vs_random = (saved_hours_vs_random / time_random) * 100

print("[BUSINESS IMPACT - K-MEANS vs RANDOM]")
print(f"-> By replacing the Random Layout with AI K-Means, the warehouse saves:")
print(f"   {saved_hours_vs_random:.2f} hours of walking time for this batch of orders.")
print(f"   Efficiency Improvement: {improvement_pct_vs_random:.2f}%\n")

# Calculate metrics compared to Traditional ABC
diff_hours_vs_abc = time_kmeans - time_abc
if diff_hours_vs_abc <= 0:
    print("[BUSINESS IMPACT - K-MEANS vs ABC]")
    print(f"-> K-Means is FASTER than ABC by {-diff_hours_vs_abc:.2f} hours.")
else:
    print("[BUSINESS IMPACT - K-MEANS vs ABC]")
    print(f"-> K-Means takes {diff_hours_vs_abc:.2f} hours MORE than ABC.")
    print("-> Trade-off: While ABC is slightly faster in raw walking time, ")
    print("   K-Means provides multi-dimensional clustering (Sales, Velocity, Frequency)")
    print("   which improves inventory management, space utilization, and batching.")

       EFFICIENCY EVALUATION (TRAVEL TIME REPORT)      
Slotting Strategy  Total Distance (Meters)  Total Travel Time (Hours)
    Random Layout                 23978120                    5550.49
       ABC Layout                 15152584                    3507.54
   K-Means Layout                 15019160                    3476.66

[BUSINESS IMPACT - K-MEANS vs RANDOM]
-> By replacing the Random Layout with AI K-Means, the warehouse saves:
   2073.83 hours of walking time for this batch of orders.
   Efficiency Improvement: 37.36%

[BUSINESS IMPACT - K-MEANS vs ABC]
-> K-Means is FASTER than ABC by 30.89 hours.
